# Therme Vals. Spatial Graph Analysis


### Peter Zumthor | Thermal Journey Network

In [2]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper
from topologicpy.Grid import Grid
from topologicpy.Graph import Graph
from topologicpy.Color import Color

c:\Users\ramyayoub\Desktop\IAAC\Master\Semester-3\Graph ML -- DOCUMENTS\Graph ML --Ramy\.gmlenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
print(Helper.Version())

The version that you are using (0.9.25) is EQUAL TO the latest version available on PyPI.


In [4]:
renderer = "vscode"

## 1. Utility functions to reset the face dictionaries and transfer dictionaries by key

In [8]:
def reset_dictionaries(shell):
    faces = Topology.Faces(shell)
    for i, f in enumerate(faces):
        d = Topology.Dictionary(f)
        keys = Dictionary.Keys(d)
        for key in keys:
            if not key == "face_id":
                d = Dictionary.RemoveKey(d, key)
        f = Topology.SetDictionary(f, d)

def transfer_dicts_by_key(topologies, selectors, key):
    dicts = {}
    for t in topologies:
        d = Topology.Dictionary(t)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            dicts[str(value)] = t
    
    for s in selectors:
        d = Topology.Dictionary(s)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            f = dicts[str(value)]
            f = Topology.SetDictionary(f, d)

## 2. Import the BREP file and clean all triangulations

In [11]:
floor_plan = Topology.ByBREPPath(r"C:\Users\ramyayoub\Desktop\IAAC\Master\Semester-3\Graph ML -- DOCUMENTS\Graph ML --Ramy\Graph ML -- Assignment\Assignment-02\01-Assets\analysis_plan.brep")
triangles = Cluster.Faces(floor_plan)
shell = Shell.ByFaces(triangles)
eb = Shell. ExternalBoundary(shell)
ib_list = Shell. InternalBoundaries (shell)
new_face = Face.ByWires(eb, ib_list)
plan_analysis = Topology.RemoveCollinearEdges (new_face)
print(plan_analysis)

In [12]:
Topology.Show(plan_analysis,
              camera=[0,0,5],
              faceColor=[50,50,250],
              faceOpacity=1,
              edgeColor="black",
              edgeWidth=5,
              backgroundColor="white",
              width=1000,
              height=800,
              renderer = renderer)

## 3. Grid Overlay --> Slicing --> Shell

In [69]:
#range() only works with integers, not floats like 0.5. so i Used numpy.arange()
#import numpy as np
b_r = Wire.BoundingRectangle(plan_analysis)
d = Topology.Dictionary(b_r)
xmin = Dictionary.ValueAtKey(d, "xmin")
xmax = Dictionary.ValueAtKey(d, "xmax")
ymin = Dictionary.ValueAtKey(d, "ymin")
ymax = Dictionary.ValueAtKey(d, "ymax")
width = Dictionary.ValueAtKey(d, "width")
length = Dictionary.ValueAtKey(d, "length")
uRange = list(range(0,int(width)+2, 2))
vRange = list(range(0,int(length)+2, 2))

#numpy
# uRange = list(np.arange(0, width + 0.8, 0.8))
# vRange = list(np.arange(0, length + 0.8, 0.8))

grid = Grid.EdgesByDistances(plan_analysis, clip=True, uRange=uRange, vRange=vRange)

In [70]:
shell = Topology.Slice(plan_analysis, grid)
faces = Topology.Faces(shell)
# Assign a sequential unique face id to reference it later (e.g. "face_21")
for i, f in enumerate(faces):
    d = Dictionary.ByKeyValue("face_id", "face_"+str(i+1))
    f = Topology.SetDictionary(f, d)

In [71]:
Topology.Show(plan_analysis, grid,
              camera=[0,0,4],
              faceColor=[50,50,250],
              faceOpacity=1,
              edgeColor="black",
              edgeWidth=3,
              showVertices=False,
              backgroundColor="white",
              width=800,
              height=600,
              renderer = renderer)

## 4. Derive navigation and analysis graphs from the shell

In [72]:
# Note: Graph nodes automatically inherit the dictionaries of the entities they 
navigation_graph = Graph.ByTopology(shell, direct=False, viaSharedTopologies=True)
analysis_graph = Graph.ByTopology(shell)

## 5. Derive and store the analysis graph vertices

In [73]:
g_verts = Graph.Vertices(analysis_graph)

In [74]:
Topology.Show(analysis_graph,
              camera=[0,0,5],
              vertexSize=4,
              vertexColor="red",
              edgeColor="lightgrey",
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)
              

## 5. Spatial Intelligence through Graph Analysis

### a. Minimum Spanning Tree

In [75]:
a1 = Graph.MinimumSpanningTree(analysis_graph)
Topology.Show(a1,
              camera=[0,0,5],
              vertexSize=12,
              vertexColor="red",
              edgeColor="lightgrey",
              edgeWidth=4,
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)


In [ ]:
dn1 = Graph.Density(analysis_graph)
dn2 = Graph.Density(a1)
print("Density 1:", dn1)
print("Density 2:", dn2)

Density 1: 0.007149934118464194
Density 2: 0.004514672686230248


In [77]:
dr1 = Graph.Diameter(analysis_graph)
dr2 = Graph.Diameter(a1 )
print("Diameter 1:", dr1)
print("Diameter 2:", dr2)

Diameter 1: 41
Diameter 2: 70
